# HMS v9s — window-shift augmentation, extra training seeds (1042, 2042)

Same experiment as v9 (`notebookda41823fab`), repeated with two more training seeds so every result in the window-shift table has a seed spread. Folds, recipe and views are unchanged; seed 42 is read from the v9 output. Results are reported per seed and for the per-EEG KL averaged over seeds.

Winning HMS pipelines feed the 50 s excerpt but shift the window by up to ±20 s during training (and sometimes at test time). On recordings that are only ~50 s long, a shifted window runs past the end of the file and is zero-padded, so "padding present" again says "this is a short-excerpt (low-vote) recording".

**Training variants** (50 s window, stage 1, same recipe and folds as before; mask = all ones, so zeros are visible as in a typical pipeline):
- `shift0` — window centred (±1 s jitter), the setting of our clean 50 s benchmark; never sees padding.
- `shift20_zero` — centre shifted uniformly by up to ±20 s, outside-recording samples zero-filled (common practice).
- `shift20_mirror` — same shifts, outside-recording samples mirror-filled (our fix).

**Evaluation views** (pooled out-of-fold, one row per EEG):
- `center` — the standard centred 50 s window; no padding is ever visible, so a learnt cue cannot act here.
- `edge_zero` vs `edge_mirror` — the centred window with its outer 10 s on each side replaced by zeros or by a mirror of the adjacent signal. Same signal removed; only the zeros look like a recording boundary. A model that learnt the cue moves toward the short-excerpt label mix (Seizure down, Other up) under zeros, and gains on no-context EEGs but loses on full-context EEGs (matched-input asymmetry).
- `tta_zero` / `tta_mirror` — average over window shifts {−20, −10, 0, +10, +20} s, as a pipeline with shift test-time augmentation would.

**Inputs:** HMS competition data + the output of the v9 notebook (`notebookda41823fab`; seed 42). GPU T4, internet off, **Save Version → Save & Run All**. About 3.5 GPU h for both seeds; runs are saved as they finish and the loop stops before `TIME_BUDGET_H`. To resume, attach this notebook's own output too.

In [1]:
import os, glob, json, math, time, random, shutil
import numpy as np
import pandas as pd

CFG = dict(
    DATA_ROOT=None,                       # None = auto-detect under /kaggle/input
    CACHE_DIR="/kaggle/working/cache",    # replaced in the v2 settings cell below
    RESULTS_DIR="/kaggle/working/results",
    FS=100,                               # target sampling rate (Hz)
    CTX_SEC=50,
    CACHE_CTX_SEC=100,
    JITTER_SEC=1.0,                       # train-time random shift of the window centre
    PATCH=25,                             # samples per token (0.25 s at 100 Hz)
    N_FOLDS=5, FOLDS=[0], SEED=42,
    MODELS=["mamba", "transformer", "gru", "cnn"],
    D_MODEL=128, N_LAYERS=4, D_STATE=64, HEADDIM=32, EXPAND=2, CHUNK=64, N_HEADS_ATTN=4,
    EPOCHS=15, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0,
    TRAIN_MIN_VOTES=0,
    HIGH_VOTE=10,                         # threshold for the high-agreement evaluation subset
    NUM_WORKERS=4, AMP=True,
    USE_OFFICIAL_MAMBA2=False,
    RUN_BENCHMARK=True,
    BENCH_CTX=[50, 100, 200, 400, 600, 1200],
)

In [2]:
import pyarrow.parquet as pq
import multiprocessing as mp
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
MP = mp.get_context("fork")   # Kaggle is Linux; fork shares globals with workers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "train_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data: Add Input → Competitions → HMS - Harmful Brain Activity Classification")

ROOT = find_root()
print("data:", ROOT)

device: cuda Tesla T4
data: /kaggle/input/competitions/hms-harmful-brain-activity-classification


## Labels and patient-level folds

In [3]:
VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
P_COLS = [f"p_{c}" for c in CLASSES]
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0   # experts labelled the central 10 s of each 50 s window

sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k
assert df.groupby("patient_id")["fold"].nunique().max() == 1, "patient leaked across folds"

print(f"rows={len(df):,}  eegs={df.eeg_id.nunique():,}  patients={df.patient_id.nunique():,}")
print(df.groupby("fold").agg(rows=("eeg_id", "size"), eegs=("eeg_id", "nunique"), patients=("patient_id", "nunique")))
print(df.drop_duplicates("eeg_id")["expert_consensus"].value_counts())

rows=106,800  eegs=17,089  patients=1,950
       rows  eegs  patients
fold                       
0     17085  2795       383
1     25502  3689       386
2     21174  3585       394
3     22141  3238       397
4     20898  3782       390
expert_consensus
Other      7196
Seizure    2785
LPD        2577
GRDA       1828
GPD        1787
LRDA        916
Name: count, dtype: int64


## Preprocessing
18-channel bipolar double banana + midline (identical to I-CARE's 18 bipolar pairs). Band-pass 0.5–40 Hz at 200 Hz, then every second sample (100 Hz; the 40 Hz low-pass prevents aliasing). Amplitudes stay in µV (no per-window z-scoring), because absolute voltage — suppression, high-amplitude discharges — is clinically informative.

In [4]:
BIPOLAR = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"),    # left temporal chain
           ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),    # right temporal chain
           ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"),    # left parasagittal chain
           ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),    # right parasagittal chain
           ("Fz", "Cz"), ("Cz", "Pz")]                                 # midline
FLIP_PERM = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11, 16, 17]   # left <-> right hemisphere
ELECTRODES = sorted({e for pair in BIPOLAR for e in pair})
N_CH = len(BIPOLAR)
SRC_FS = 200
SOS = butter(4, [0.5, 40.0], btype="band", fs=SRC_FS, output="sos")
CLIP_UV = 500.0

def load_bipolar(path):
    d = pd.read_parquet(path, columns=ELECTRODES)
    d = d.interpolate(limit_direction="both").fillna(0.0)
    a = d.to_numpy(np.float32)
    col = {c: i for i, c in enumerate(ELECTRODES)}
    x = np.stack([a[:, col[p]] - a[:, col[q]] for p, q in BIPOLAR], axis=1)
    x = sosfiltfilt(SOS, x, axis=0)
    return np.clip(x, -CLIP_UV, CLIP_UV).astype(np.float32)

def _nrows(eid):
    return pq.ParquetFile(f"{ROOT}/train_eegs/{eid}.parquet").metadata.num_rows

def plan_cache(df, half_sec):
    g = df.groupby("eeg_id")["center"].agg(["min", "max"]).reset_index()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        n = pool.map(_nrows, g["eeg_id"].tolist(), chunksize=64)
    g["n_src"] = n
    s0 = np.floor((g["min"] - half_sec) * SRC_FS).clip(lower=0).astype(np.int64)
    s0 -= s0 % 2
    s1 = np.minimum(g["n_src"], np.ceil((g["max"] + half_sec) * SRC_FS)).astype(np.int64)
    g["s0"], g["length"] = s0, ((s1 - s0 + 1) // 2).clip(lower=0)
    g["offset"] = np.concatenate([[0], np.cumsum(g["length"].to_numpy())[:-1]])
    return g[["eeg_id", "n_src", "s0", "length", "offset"]]

def _build_one(args):
    eid, s0, length = args
    x = load_bipolar(f"{ROOT}/train_eegs/{eid}.parquet")[s0::2][:length]
    out = np.zeros((length, N_CH), np.float16)
    out[:len(x)] = x
    return out

def build_cache(df):
    meta_path = f"{CFG['CACHE_DIR']}/meta.json"
    want = dict(cache_ctx=CFG["CACHE_CTX_SEC"], jitter=CFG["JITTER_SEC"], fs=CFG["FS"], n_eegs=int(df.eeg_id.nunique()))
    if os.path.exists(meta_path) and json.load(open(meta_path)) == want:
        print("cache found, reusing")
        return pd.read_csv(f"{CFG['CACHE_DIR']}/index.csv")
    half = CFG["CACHE_CTX_SEC"] / 2 + CFG["JITTER_SEC"]
    idx = plan_cache(df, half)
    total = int(idx["length"].sum())
    need = total * N_CH * 2
    free = shutil.disk_usage(CFG["CACHE_DIR"]).free
    print(f"cache: {total/CFG['FS']/3600:,.1f} h of EEG, {need/1e9:.2f} GB needed, {free/1e9:.1f} GB free")
    if need > 0.9 * free:
        raise RuntimeError("Not enough disk: lower CACHE_CTX_SEC or point CACHE_DIR at a larger disk")
    mm = np.lib.format.open_memmap(f"{CFG['CACHE_DIR']}/signals.npy", mode="w+", dtype=np.float16, shape=(total, N_CH))
    jobs = list(zip(idx["eeg_id"], idx["s0"], idx["length"]))
    t0 = time.time()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        for i, (x, off) in enumerate(zip(pool.imap(_build_one, jobs, chunksize=16), idx["offset"])):
            mm[off:off + len(x)] = x
            if i % 2000 == 0:
                print(f"  {i:,}/{len(jobs):,}  {time.time()-t0:.0f}s")
    mm.flush(); del mm
    idx.to_csv(f"{CFG['CACHE_DIR']}/index.csv", index=False)
    json.dump(want, open(meta_path, "w"))
    print(f"cache built in {time.time()-t0:.0f}s")
    return idx

## Dataset
Each item is a `CTX_SEC` window centred on the labelled 10 s. Positions outside the recording are zero-filled and flagged in a mask; the SSM treats masked steps as identity transitions (Δ = 0), the Transformer ignores them via key-padding, the readout never pools them.
Training samples one random sub-window per EEG per epoch, so long recordings with many overlapping sub-windows don't dominate. Validation uses the middle sub-window of each EEG (the competition test set also has one row per EEG).

In [5]:
class HMSDataset(Dataset):
    """Self-contained (no notebook globals in __getitem__), so it works with any worker start method."""
    def __init__(self, rows, ctx_sec, train):
        assert ctx_sec <= CFG["CACHE_CTX_SEC"], "CTX_SEC exceeds the cached context; rebuild the cache"
        self.eid = rows["eeg_id"].to_numpy()
        self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32)
        self.y = rows["y"].to_numpy()
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs)
        assert self.n % CFG["PATCH"] == 0
        self.train, self.jitter, self.flip = train, CFG["JITTER_SEC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"
        self.mm = None

    def __len__(self):
        return len(self.eid)

    def __getitem__(self, i):
        if self.mm is None:                      # open lazily so each worker gets its own handle
            self.mm = np.load(self.path, mmap_mode="r")
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i]
        if self.train:
            c += np.random.uniform(-self.jitter, self.jitter)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        x = np.zeros((self.n, self.n_ch), np.float32)
        m = np.zeros(self.n, np.float32)
        a, b = max(i0, 0), min(i0 + self.n, length)
        if b > a:
            x[a - i0:b - i0] = self.mm[off + a:off + b]
            m[a - i0:b - i0] = 1.0
        x = x.T / 50.0                           # ~unit scale while keeping absolute amplitude
        if self.train:
            if np.random.rand() < 0.5:
                x = x[self.flip]
            x = x * np.random.uniform(0.9, 1.1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(m), torch.from_numpy(self.p[i]), int(self.y[i])

class OnePerEEG(Sampler):
    def __init__(self, eids, seed):
        order = np.argsort(eids, kind="stable")
        _, starts = np.unique(eids[order], return_index=True)
        self.groups = np.split(order, starts[1:])
        self.seed, self.epoch = seed, 0
    def __len__(self):
        return len(self.groups)
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
        idx = np.array([g[rng.integers(len(g))] for g in self.groups])
        rng.shuffle(idx)
        return iter(idx.tolist())

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]

## Models
All models: `Stem` (conv + 0.25 s patching → tokens) → sequence mixer → readout (pool over the central 10 s ⊕ pool over the full context) → 6 logits. Only the mixer differs.

The SSM is a bidirectional Mamba-2 block. `ssd()` is the chunked state-space-duality algorithm (exact); `ssd_reference()` is the plain recurrence used to verify it. `Mamba2.step()` is the recurrent form used for streaming.

In [6]:
from torch.utils.checkpoint import checkpoint

def run_block(blk, *args):
    if CFG["GRAD_CKPT"] and blk.training and torch.is_grad_enabled():
        return checkpoint(blk, *args, use_reentrant=False)
    return blk(*args)

def segsum(x):
    """x: (..., T) log-decays -> (..., T, T) with out[i, j] = sum_{k=j+1..i} x_k for i >= j, else -inf."""
    T = x.size(-1)
    x = x[..., None].expand(*x.shape, T)
    low = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), -1)
    x = x.masked_fill(~low, 0)
    s = torch.cumsum(x, dim=-2)
    keep = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), 0)
    return s.masked_fill(~keep, -torch.inf)

def ssd(X, A, B, C, chunk):
    """Selective SSM, chunked SSD form.  h_t = exp(A_t) h_{t-1} + X_t B_t^T,  y_t = h_t C_t
    X: (b, T, h, p) input already multiplied by dt;  A: (b, T, h) = dt * A (<= 0);
    B, C: (b, T, n), shared by all heads (Mamba-2 with ngroups = 1)."""
    b, T, h, p = X.shape
    pad = (-T) % chunk
    if pad:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        B, C = F.pad(B, (0, 0, 0, pad)), F.pad(C, (0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
    c = (T + pad) // chunk
    X = X.reshape(b, c, chunk, h, p)
    B = B.reshape(b, c, chunk, -1)
    C = C.reshape(b, c, chunk, -1)
    A = A.reshape(b, c, chunk, h).permute(0, 3, 1, 2)            # (b, h, c, l)
    A_cs = torch.cumsum(A, dim=-1)
    L = torch.exp(segsum(A))                                     # (b, h, c, l, s)
    CB = torch.einsum("bcln,bcsn->bcls", C, B)
    Y_diag = torch.einsum("bcls,bhcls,bcshp->bclhp", CB, L, X)
    decay_states = torch.exp(A_cs[..., -1:] - A_cs)
    states = torch.einsum("bcln,bhcl,bclhp->bchpn", B, decay_states, X)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)
    decay_chunk = torch.exp(segsum(F.pad(A_cs[..., -1], (1, 0))))
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]
    Y_off = torch.einsum("bcln,bchpn,bhcl->bclhp", C, states, torch.exp(A_cs))
    return (Y_diag + Y_off).reshape(b, c * chunk, h, p)[:, :T]

def ssd_reference(X, A, B, C):
    b, T, h, p = X.shape
    state = X.new_zeros(b, h, p, B.size(-1))
    ys = []
    for t in range(T):
        state = torch.exp(A[:, t])[..., None, None] * state + X[:, t, :, :, None] * B[:, t, None, None, :]
        ys.append(torch.einsum("bhpn,bn->bhp", state, C[:, t]))
    return torch.stack(ys, 1)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

class Mamba2(nn.Module):
    """Mamba-2 mixer (pure PyTorch). Masked steps get dt = 0: no decay, no input -> state passes through unchanged."""
    def __init__(self, d, d_state, headdim, expand, chunk, d_conv=4):
        super().__init__()
        self.di, self.n, self.p, self.chunk = expand * d, d_state, headdim, chunk
        self.h = self.di // headdim
        self.in_proj = nn.Linear(d, 2 * self.di + 2 * d_state + self.h, bias=False)
        cd = self.di + 2 * d_state
        self.conv = nn.Conv1d(cd, cd, d_conv, groups=cd, padding=d_conv - 1)
        dt = torch.exp(torch.rand(self.h) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))     # inverse softplus
        self.A_log = nn.Parameter(torch.log(torch.empty(self.h).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.h))
        self.norm = RMSNorm(self.di)
        self.out_proj = nn.Linear(self.di, d, bias=False)

    def forward(self, u, m):
        b, T, _ = u.shape
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        xBC = xBC * m[..., None]
        xBC = F.silu(self.conv(xBC.transpose(1, 2))[..., :T].transpose(1, 2))
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        with torch.autocast(device_type=u.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias) * m[..., None]
            A = -torch.exp(self.A_log.float())
            x = x.float().reshape(b, T, self.h, self.p)
            y = ssd(x * dt[..., None], A * dt, Bm.float(), Cm.float(), self.chunk) + x * self.D[:, None]
        y = y.reshape(b, T, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z)))

    def init_state(self, b, device):
        cd = self.di + 2 * self.n
        return (torch.zeros(b, cd, self.conv.kernel_size[0], device=device),
                torch.zeros(b, self.h, self.p, self.n, device=device))

    def step(self, u, state):
        """One token, u: (b, d). Fixed-size state: conv window (b, cd, d_conv) + SSM state (b, h, p, n)."""
        conv_st, ssm_st = state
        b = u.size(0)
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        conv_st = torch.roll(conv_st, -1, dims=-1)
        conv_st[..., -1] = xBC.float()
        xBC = F.silu((conv_st * self.conv.weight[:, 0, :]).sum(-1) + self.conv.bias)
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        dt = F.softplus(dt.float() + self.dt_bias)
        dA = torch.exp(dt * -torch.exp(self.A_log.float()))
        x = x.float().reshape(b, self.h, self.p)
        ssm_st = ssm_st * dA[..., None, None] + (x * dt[..., None])[..., None] * Bm.float()[:, None, None, :]
        y = torch.einsum("bhpn,bn->bhp", ssm_st, Cm.float()) + x * self.D[:, None]
        y = y.reshape(b, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z))), (conv_st, ssm_st)

class OfficialMamba2(nn.Module):
    def __init__(self, d, d_state, headdim, expand, chunk):
        super().__init__()
        from mamba_ssm import Mamba2 as M2
        self.m = M2(d_model=d, d_state=d_state, headdim=headdim, expand=expand, chunk_size=chunk)
    def forward(self, u, m):
        return self.m(u * m[..., None])

class BiSSMBlock(nn.Module):
    def __init__(self, d, bidir=True):
        super().__init__()
        cls = OfficialMamba2 if CFG["USE_OFFICIAL_MAMBA2"] else Mamba2
        mk = lambda: cls(d, CFG["D_STATE"], CFG["HEADDIM"], CFG["EXPAND"], CFG["CHUNK"])
        self.norm, self.fwd = nn.LayerNorm(d), mk()
        self.bwd = mk() if bidir else None
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = self.norm(h)
        mf = m.float()
        y = self.fwd(x, mf)
        if self.bwd is not None:
            y = y + self.bwd(x.flip(1), mf.flip(1)).flip(1)
        return h + self.drop(y)

class SSMMixer(nn.Module):
    """bidir=True: L bidirectional blocks. bidir=False: 2L forward-only blocks (same parameter count), causal over the sequence."""
    def __init__(self, d, L, bidir=True):
        super().__init__()
        self.blocks = nn.ModuleList([BiSSMBlock(d, bidir) for _ in range(L if bidir else 2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

def sinusoid(n, d, device):
    pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, device=device, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe

class TransformerMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d, CFG["N_HEADS_ATTN"], 4 * d, dropout=0.1, activation="gelu",
                                                                batch_first=True, norm_first=True) for _ in range(L)])
    def forward(self, h, m):
        h = h + sinusoid(h.size(1), h.size(2), h.device).to(h.dtype)
        kpm = ~m
        for layer in self.layers:
            h = run_block(layer, h, None, kpm)
        return h

class GRUMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.rnn = nn.GRU(d, d, L, batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(2 * d, d)
    def forward(self, h, m):
        return self.proj(run_block(self.rnn, h * m[..., None])[0])

class ConvBlock(nn.Module):
    def __init__(self, d, dil):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.dw = nn.Conv1d(d, d, 9, padding=4 * dil, dilation=dil, groups=d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = (self.norm(h) * m[..., None]).transpose(1, 2)
        return h + self.drop(self.mlp(self.dw(x).transpose(1, 2)))

class CNNMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__(); self.blocks = nn.ModuleList([ConvBlock(d, 2 ** (i % 6)) for i in range(2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

MIXERS = dict(mamba=SSMMixer, mamba_causal=lambda d, L: SSMMixer(d, L, bidir=False),
              transformer=TransformerMixer, gru=GRUMixer, cnn=CNNMixer)

class Stem(nn.Module):
    def __init__(self, d, patch):
        super().__init__()
        self.patch = patch
        self.c1 = nn.Conv1d(N_CH, d, 7, padding=3)
        self.c2 = nn.Conv1d(d, d, patch, stride=patch)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, m):
        h = self.c2(F.gelu(self.c1(x * m[:, None]))).transpose(1, 2)
        tm = F.avg_pool1d(m[:, None], self.patch, self.patch)[:, 0] > 0.5
        return self.norm(h) * tm[..., None], tm

class Net(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        d = CFG["D_MODEL"]
        self.stem = Stem(d, CFG["PATCH"])
        self.mixer = MIXERS[mixer](d, CFG["N_LAYERS"])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, len(CLASSES)))
        self.half_center = int(5 * CFG["FS"] / CFG["PATCH"])     # tokens in 5 s

    def forward(self, x, m):
        h, tm = self.stem(x, m)
        h = self.norm(self.mixer(h, tm))
        c = h.size(1) // 2
        cm = torch.zeros_like(tm)
        cm[:, max(c - self.half_center, 0):c + self.half_center] = True
        cm &= tm
        def pool(mask):
            w = mask.to(h.dtype)[..., None]
            return (h * w).sum(1) / w.sum(1).clamp(min=1)
        return self.head(torch.cat([pool(cm), pool(tm)], dim=-1))

def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## v9 settings

In [7]:
CFG.update(dict(
    CACHE_CTX_SEC=90, CACHE_DIR="/tmp/hms_cache90", RESULTS_DIR="/kaggle/working/results_v9",
    PREV_GLOBS=["/kaggle/input/**/results_v9/*.json", "/kaggle/input/**/results_v9/*_valpred.npy"],
    CTX=200,                      # context bins are defined on a 200 s window, as in the paper
    WIN=50, SHIFT=20.0, EDGE=10.0, TTA_SHIFTS=[-20, -10, 0, 10, 20],
    N_FOLDS=5, EPOCHS=6, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0, TOKENS_PER_BATCH=25600, JITTER_SEC=1.0,
    DROPOUT=0.1, GRAD_CKPT=False, USE_OFFICIAL_MAMBA2=False,
    # (model, variant) in priority order: the shortcut test needs shift20_zero and shift0; shift20_mirror is the fix
    # (model, variant, training seed) in priority order; seed 42 = the v9 runs (attach that output)
    RUNS=[(m, v, s) for s in (1042, 2042) for m, v in [("mamba", "shift20_zero"), ("mamba", "shift0"), ("gru", "shift20_zero"), ("gru", "shift0"),
                                                      ("mamba", "shift20_mirror"), ("gru", "shift20_mirror")]],
    SEEDS=[42, 1042, 2042],
    EST_MIN_PER_FOLD=dict(mamba=5.5, gru=1.5), TIME_BUDGET_H=5.0,
    B=4000, MIN_N=20, T_GRID=list(np.round(np.linspace(0.5, 3.0, 51), 3)),
    CTX_BINS=[-1, 10, 140, 1000], CTX_LABELS=["none (<10 s)", "partial (10-140 s)", "full (>=140 s)"],
    VOTE_GROUPS=[(1, 2, "1-2 votes"), (3, 9, "3-9 votes"), (10, 10_000, ">=10 votes")],
))
CFG_OVERRIDES = {}
CFG.update(CFG_OVERRIDES)
assert CFG["CACHE_CTX_SEC"] / 2 >= CFG["WIN"] / 2 + CFG["SHIFT"], "cache must cover every shifted window, or padding would appear inside the recording"
T_START = time.time()
os.makedirs(CFG["CACHE_DIR"], exist_ok=True); os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
n_copied = 0
for pat in CFG["PREV_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        dst = os.path.join(CFG["RESULTS_DIR"], os.path.basename(fp))
        if not os.path.exists(dst):
            shutil.copy(fp, dst); n_copied += 1
print(f"resumed {n_copied} files from earlier v9 outputs")
PREDS = {}

resumed 360 files from earlier v9 outputs


## Per-EEG and per-row context availability

In [8]:
cache_idx = build_cache(df)
CACHE = {int(r.eeg_id): (int(r.offset), int(r.s0), int(r.length)) for r in cache_idx.itertuples()}
DUR = dict(zip(cache_idx["eeg_id"].astype(int), cache_idx["n_src"] / SRC_FS))

def extra_context(center, dur, ctx=CFG["CTX"]):
    half = ctx / 2
    left = np.minimum(center, half); right = np.clip(np.minimum(dur - center, half), 0, None)
    return left + right - np.minimum(center, 25) - np.clip(np.minimum(dur - center, 25), 0, None)

df["dur_s"] = df["eeg_id"].map(DUR)
df["extra_s"] = extra_context(df["center"].to_numpy(), df["dur_s"].to_numpy())
df["ctx_bin"] = pd.cut(df["extra_s"], CFG["CTX_BINS"], labels=CFG["CTX_LABELS"]).astype(str)

VA = {f: middle_row_per_eeg(df[df.fold == f]).reset_index(drop=True) for f in range(CFG["N_FOLDS"])}
OOF = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])], ignore_index=True)
T = OOF[P_COLS].to_numpy(np.float64)
NV = OOF["n_votes"].to_numpy(); PID = OOF["patient_id"].to_numpy(); FOLD = OOF["fold"].to_numpy()
OOF["vote_group"] = ""
for lo, hi, name in CFG["VOTE_GROUPS"]:
    OOF.loc[(NV >= lo) & (NV <= hi), "vote_group"] = name
OOF["label"] = np.array(CLASSES)[OOF["y"].to_numpy()]
VG, CB = OOF["vote_group"].to_numpy(), OOF["ctx_bin"].to_numpy()
print(pd.crosstab(OOF["vote_group"], OOF["ctx_bin"], margins=True).to_string())

def per_eeg_kl(P, eps=1e-15):
    return (T * (np.log(T + eps) - np.log(P + eps))).sum(1)

def load_oof(model, ctx, stage):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = PREDS.get(f"{model}_ctx{ctx}_f{f}_{stage}_valpred.npy")
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

UP, INV = np.unique(PID, return_inverse=True)
W = np.random.default_rng(CFG["SEED"]).multinomial(len(UP), np.full(len(UP), 1.0 / len(UP)), size=CFG["B"]).astype(np.float32)

def boot(d, msk):
    if msk.sum() == 0:
        return (float("nan"),) * 4
    s_ = np.bincount(INV[msk], weights=d[msk], minlength=len(UP)).astype(np.float32)
    n_ = np.bincount(INV[msk], minlength=len(UP)).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (W @ s_) / (W @ n_)
    r = r[np.isfinite(r)]
    if len(r) == 0:
        return float(d[msk].mean()), float("nan"), float("nan"), float("nan")
    lo, hi = np.percentile(r, [2.5, 97.5])
    return float(d[msk].mean()), lo, hi, max(min(1.0, 2 * min((r <= 0).mean(), (r >= 0).mean())), 1.0 / len(r))

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    idx = np.argsort(np.where(ok, p, np.inf)); k = ok.sum(); run = 0.0
    for i_, j in enumerate(idx[:k]):
        run = max(run, min(1.0, (k - i_) * p[j])); out[j] = run
    return out

SUBSETS = {"all EEGs": np.ones(len(OOF), bool)}
for b in CFG["CTX_LABELS"]:
    SUBSETS[f"all | {b}"] = CB == b
for g in ["1-2 votes", "3-9 votes", ">=10 votes"]:
    SUBSETS[f"{g} | all"] = VG == g
    for b in CFG["CTX_LABELS"]:
        SUBSETS[f"{g} | {b}"] = (VG == g) & (CB == b)

cache: 380.3 h of EEG, 4.93 GB needed, 1038.2 GB free
  0/17,089  1s
  2,000/17,089  21s
  4,000/17,089  42s
  6,000/17,089  63s
  8,000/17,089  83s
  10,000/17,089  104s
  12,000/17,089  124s
  14,000/17,089  144s
  16,000/17,089  165s
cache built in 178s
ctx_bin     full (>=140 s)  none (<10 s)  partial (10-140 s)    All
vote_group                                                         
1-2 votes               25          4526                 296   4847
3-9 votes              297          2298                4047   6642
>=10 votes             169          2648                2783   5600
All                    491          9472                7126  17089


In [9]:
def param_groups(model):
    decay, no_decay = [], []
    for n_, p in model.named_parameters():
        (no_decay if p.ndim < 2 or n_.endswith(("A_log", "D", "dt_bias")) else decay).append(p)
    return [dict(params=decay, weight_decay=CFG["WD"]), dict(params=no_decay, weight_decay=0.0)]

def loader(ds, batch, sampler=None, drop_last=False):
    return DataLoader(ds, batch, sampler=sampler, shuffle=False, drop_last=drop_last, num_workers=CFG["NUM_WORKERS"],
                      pin_memory=DEV.type == "cuda", persistent_workers=CFG["NUM_WORKERS"] > 0,
                      multiprocessing_context=MP if CFG["NUM_WORKERS"] > 0 else None)

@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for x, m, _, _ in dl:
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=CFG["AMP"] and DEV.type == "cuda"):
            out.append(torch.softmax(model(x.to(DEV), m.to(DEV)).float(), -1).cpu())
    return torch.cat(out).numpy()

## Shifted-window dataset, training and evaluation views

In [10]:
class ShiftDataset(Dataset):
    """50 s window around the label centre. train: centre shifted by U(-shift, shift) (+ jitter), outside-recording samples
    filled with zeros or a mirror. eval: view = center | edge_zero | edge_mirror | tta_<fill>_<shift>. Mask is all ones."""
    def __init__(self, rows, train, shift=0.0, fill="zero", view="center"):
        self.eid = rows["eeg_id"].to_numpy(); self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32)
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(CFG["WIN"] * self.fs); self.e = int(CFG["EDGE"] * self.fs)
        self.train, self.shift, self.fill, self.view = train, shift, fill, view
        self.jitter, self.flip = CFG["JITTER_SEC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"; self.mm = None
    def __len__(self):
        return len(self.eid)
    def window(self, i, c, fill):
        off, s0, length = self.cache[int(self.eid[i])]
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        a, b = max(i0, 0), min(i0 + self.n, length)
        if b <= a:
            return np.zeros((self.n, self.n_ch), np.float32)
        real = np.asarray(self.mm[off + a:off + b], dtype=np.float32)
        lo, hi = a - i0, self.n - (b - i0)
        if lo == 0 and hi == 0:
            return real
        if fill == "mirror":
            return np.pad(real, ((lo, hi), (0, 0)), mode="symmetric")
        return np.pad(real, ((lo, hi), (0, 0)))
    def __getitem__(self, i):
        if self.mm is None:
            self.mm = np.load(self.path, mmap_mode="r")
        rng = np.random.default_rng()
        c = self.center[i]
        if self.train:
            c += rng.uniform(-self.jitter, self.jitter) + (rng.uniform(-self.shift, self.shift) if self.shift > 0 else 0.0)
            x = self.window(i, c, self.fill)
        elif self.view.startswith("tta_"):
            _, fill, s = self.view.split("_")
            x = self.window(i, c + float(s), fill)
        else:
            x = self.window(i, c, "zero").copy()
            if self.view == "edge_zero":
                x[:self.e] = 0.0; x[-self.e:] = 0.0
            elif self.view == "edge_mirror":
                x[:self.e] = x[self.e:2 * self.e][::-1]; x[-self.e:] = x[-2 * self.e:-self.e][::-1]
        x = x.T / 50.0
        if self.train:
            if rng.random() < 0.5:
                x = x[self.flip]
            x = x * rng.uniform(0.9, 1.1)
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)), torch.ones(self.n), torch.from_numpy(self.p[i]), torch.tensor(1.0))

def views():
    v = ["center", "edge_zero", "edge_mirror"]
    for fill in ("zero", "mirror"):
        v += [f"tta_{fill}_{s}" for s in CFG["TTA_SHIFTS"] if s != 0]
    return v

# sanity: the centred view of a real EEG is inside the recording (no padding), the edge views differ only in the outer 10 s
_r = df.drop_duplicates("eeg_id").head(64)
_d = {v: ShiftDataset(_r, False, view=v) for v in ["center", "edge_zero", "edge_mirror"]}
for i in range(len(_r)):
    xc, xz, xm = (_d[v][i][0].numpy() for v in ["center", "edge_zero", "edge_mirror"])
    e = int(CFG["EDGE"] * CFG["FS"])
    assert np.allclose(xc[:, e:-e], xz[:, e:-e]) and np.allclose(xc[:, e:-e], xm[:, e:-e])
    assert np.abs(xz[:, :e]).max() == 0
print("view sanity checks passed")

def run_tag(model, variant, fold, seed=42):
    return f"{model}_{variant}_ctx{CFG['WIN']}_f{fold}" + ("" if seed == 42 else f"_seed{seed}")

def train_shift(name, variant, fold, seed):
    tag = run_tag(name, variant, fold, seed)
    seed_all(seed + fold)
    shift = 0.0 if variant == "shift0" else CFG["SHIFT"]
    fill = "mirror" if variant.endswith("mirror") else "zero"
    tr = df[df.fold != fold].reset_index(drop=True)
    tok = CFG["WIN"] * CFG["FS"] // CFG["PATCH"]
    micro = max(1, min(CFG["BATCH"], CFG["TOKENS_PER_BATCH"] // tok)); accum = math.ceil(CFG["BATCH"] / micro)
    dl = loader(ShiftDataset(tr, True, shift, fill), micro, OnePerEEG(tr["eeg_id"].to_numpy(), seed + fold), drop_last=True)
    model = Net(name).to(DEV)
    opt = torch.optim.AdamW(param_groups(model), lr=CFG["LR"])
    total = max(1, CFG["EPOCHS"] * (len(dl) // accum)); warm = max(1, int(CFG["WARMUP_FRAC"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    use_amp = CFG["AMP"] and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"\n=== {tag} | shift ±{shift:.0f} s, fill {fill} | micro {micro} x accum {accum}")
    t_run, hist = time.time(), []
    for ep in range(CFG["EPOCHS"]):
        model.train(); t0 = time.time(); tl = nb = 0.0
        opt.zero_grad(set_to_none=True)
        for i, (x, m, p, _) in enumerate(dl):
            x, m, p = x.to(DEV, non_blocking=True), m.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x, m)
            loss = F.kl_div(F.log_softmax(logits.float(), -1), p, reduction="batchmean")
            scaler.scale(loss / accum).backward()
            tl += loss.item(); nb += 1
            if (i + 1) % accum == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CFG["CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), sec=time.time() - t0))
        print(f"  {tag} ep {ep} | train KL {tl/max(nb,1):.4f} | {time.time()-t0:.0f}s")
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"{CFG['RESULTS_DIR']}/{tag}.pt")
    for v in views():
        np.save(f"{CFG['RESULTS_DIR']}/{tag}_{v}_valpred.npy", predict(model, loader(ShiftDataset(VA[fold], False, view=v), micro * 2)))
    json.dump(dict(model=name, variant=variant, fold=fold, seed=seed, minutes=(time.time() - t_run) / 60, history=hist),
              open(f"{CFG['RESULTS_DIR']}/{tag}.json", "w"), indent=1)
    print(f"=== {tag} done in {(time.time()-t_run)/60:.1f} min")
    del model
    if DEV.type == "cuda": torch.cuda.empty_cache()

def done(name, variant, fold, seed):
    return all(os.path.exists(f"{CFG['RESULTS_DIR']}/{run_tag(name, variant, fold, seed)}_{v}_valpred.npy") for v in views())

PLAN = [(m_, v_, f_, s_) for m_, v_, s_ in CFG["RUNS"] for f_ in range(CFG["N_FOLDS"])]
todo = [r for r in PLAN if not done(*r)]
print(f"{len(PLAN)} runs planned, {len(PLAN) - len(todo)} already done, ~{sum(CFG['EST_MIN_PER_FOLD'].get(r[0], 5) for r in todo)/60:.1f} GPU-h left")
for m_, v_, f_, s_ in todo:
    used = (time.time() - T_START) / 3600
    if used + CFG["EST_MIN_PER_FOLD"].get(m_, 5) / 60 > CFG["TIME_BUDGET_H"]:
        print(f"time budget: stopping before {run_tag(m_, v_, f_, s_)} ({used:.2f} h used). Attach this output and rerun to continue.")
        break
    train_shift(m_, v_, f_, s_)

view sanity checks passed
60 runs planned, 0 already done, ~3.5 GPU-h left

=== mamba_shift20_zero_ctx50_f0_seed1042 | shift ±20 s, fill zero | micro 64 x accum 1
  mamba_shift20_zero_ctx50_f0_seed1042 ep 0 | train KL 0.9604 | 49s
  mamba_shift20_zero_ctx50_f0_seed1042 ep 1 | train KL 0.7010 | 46s
  mamba_shift20_zero_ctx50_f0_seed1042 ep 2 | train KL 0.6202 | 46s
  mamba_shift20_zero_ctx50_f0_seed1042 ep 3 | train KL 0.5643 | 46s
  mamba_shift20_zero_ctx50_f0_seed1042 ep 4 | train KL 0.5186 | 46s
  mamba_shift20_zero_ctx50_f0_seed1042 ep 5 | train KL 0.4945 | 46s
=== mamba_shift20_zero_ctx50_f0_seed1042 done in 5.3 min

=== mamba_shift20_zero_ctx50_f1_seed1042 | shift ±20 s, fill zero | micro 64 x accum 1
  mamba_shift20_zero_ctx50_f1_seed1042 ep 0 | train KL 0.9683 | 43s
  mamba_shift20_zero_ctx50_f1_seed1042 ep 1 | train KL 0.7121 | 43s
  mamba_shift20_zero_ctx50_f1_seed1042 ep 2 | train KL 0.6288 | 43s
  mamba_shift20_zero_ctx50_f1_seed1042 ep 3 | train KL 0.5703 | 43s
  mamba_shif

## Results
Calibrated with a cross-fitted temperature fitted on each model's own `center` view and reused for its other views; patient-level cluster bootstrap; Holm over each table.

In [11]:
def load_view(name, variant, view, seed=42):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = f"{CFG['RESULTS_DIR']}/{run_tag(name, variant, f, seed)}_{view}_valpred.npy"
        if not os.path.exists(fp):
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def temper(P, t):
    L_ = np.log(np.clip(P, 1e-12, 1)) / t; L_ -= L_.max(1, keepdims=True); E = np.exp(L_)
    return E / E.sum(1, keepdims=True)

def fit_T(P):
    return {f: min(CFG["T_GRID"], key=lambda t: per_eeg_kl(temper(P, t))[FOLD != f].mean()) for f in range(CFG["N_FOLDS"])}

def apply_T(P, Ts):
    out = np.empty_like(P)
    for f, t in Ts.items():
        out[FOLD == f] = temper(P[FOLD == f], t)
    return out

RAW, KLC, TS = {}, {}, {}
VARIANTS = list(dict.fromkeys((m_, v_) for m_, v_, _ in CFG["RUNS"]))
for m_, v_ in VARIANTS:
    for s_ in CFG["SEEDS"]:
        Pc = load_view(m_, v_, "center", s_)
        if Pc is None:
            continue
        key = (m_, v_, s_)
        TS[key] = fit_T(Pc)
        for view in views():
            P = load_view(m_, v_, view, s_)
            if P is None:
                continue
            RAW[key + (view,)] = P
            KLC[key + (view,)] = per_eeg_kl(apply_T(P, TS[key]))
        for fill in ("zero", "mirror"):
            Ps = [Pc] + [load_view(m_, v_, f"tta_{fill}_{sh}", s_) for sh in CFG["TTA_SHIFTS"] if sh != 0]
            if all(P is not None for P in Ps):
                P = np.mean(Ps, 0); RAW[key + (f"tta_{fill}",)] = P
                KLC[key + (f"tta_{fill}",)] = per_eeg_kl(apply_T(P, TS[key]))
    # seed-averaged per-EEG KL (and probabilities) over the seeds that have every view
    have = [s_ for s_ in CFG["SEEDS"] if (m_, v_, s_) in TS]
    if len(have) > 1:
        for view in views() + ["tta_zero", "tta_mirror"]:
            if all((m_, v_, s_, view) in KLC for s_ in have):
                KLC[(m_, v_, "mean", view)] = np.mean([KLC[(m_, v_, s_, view)] for s_ in have], 0)
                RAW[(m_, v_, "mean", view)] = np.mean([RAW[(m_, v_, s_, view)] for s_ in have], 0)
        TS[(m_, v_, "mean")] = {f: float(np.mean([TS[(m_, v_, s_)][f] for s_ in have])) for f in range(CFG["N_FOLDS"])}
    print(f"{m_} {v_}: seeds {have}")
print("available:", sorted({k[:3] for k in KLC}, key=str))
NONE, FULL = CB == CFG["CTX_LABELS"][0], CB == CFG["CTX_LABELS"][2]
ALL = np.ones(len(OOF), bool)

rows = []
for (m_, v_, s_), _ in TS.items():
    k = KLC[(m_, v_, s_, "center")]
    rows.append(dict(model=m_, variant=v_, seed=s_, temperature=np.mean(list(TS[(m_, v_, s_)].values())), kl_center_all=k.mean(), kl_center_ge10=k[NV >= 10].mean(),
                     **{f"kl_center | {b}": k[CB == b].mean() for b in CFG["CTX_LABELS"]}))
t1 = pd.DataFrame(rows); t1.to_csv(f"{CFG['RESULTS_DIR']}/v9_center_kl.csv", index=False)
print("### Centred 50 s evaluation (the standard benchmark view), calibrated KL\n" + t1.round(4).to_string(index=False))

mamba shift20_zero: seeds [42, 1042, 2042]
mamba shift0: seeds [42, 1042, 2042]
gru shift20_zero: seeds [42, 1042, 2042]
gru shift0: seeds [42, 1042, 2042]
mamba shift20_mirror: seeds [42, 1042, 2042]
gru shift20_mirror: seeds [42, 1042, 2042]
available: [('gru', 'shift0', 'mean'), ('gru', 'shift0', 1042), ('gru', 'shift0', 2042), ('gru', 'shift0', 42), ('gru', 'shift20_mirror', 'mean'), ('gru', 'shift20_mirror', 1042), ('gru', 'shift20_mirror', 2042), ('gru', 'shift20_mirror', 42), ('gru', 'shift20_zero', 'mean'), ('gru', 'shift20_zero', 1042), ('gru', 'shift20_zero', 2042), ('gru', 'shift20_zero', 42), ('mamba', 'shift0', 'mean'), ('mamba', 'shift0', 1042), ('mamba', 'shift0', 2042), ('mamba', 'shift0', 42), ('mamba', 'shift20_mirror', 'mean'), ('mamba', 'shift20_mirror', 1042), ('mamba', 'shift20_mirror', 2042), ('mamba', 'shift20_mirror', 42), ('mamba', 'shift20_zero', 'mean'), ('mamba', 'shift20_zero', 1042), ('mamba', 'shift20_zero', 2042), ('mamba', 'shift20_zero', 42)]
### Cent

### Did shift-augmented training learn the boundary cue?
Matched-input asymmetry of the edge probe: relative KL change from `edge_mirror` to `edge_zero` on no-context EEGs minus the same on full-context EEGs. Zero for a model that does not read the zeros as a recording boundary; negative when zeros help on short-excerpt EEGs and hurt on long ones. The probability shift is `edge_zero − edge_mirror` on full-context EEGs (the shortcut direction found earlier: Seizure down, Other up).

In [12]:
def rel_reps(ka, kb, msk):
    s_a = np.bincount(INV[msk], weights=(ka - kb)[msk], minlength=len(UP)).astype(np.float32)
    s_b = np.bincount(INV[msk], weights=kb[msk], minlength=len(UP)).astype(np.float32)
    return (W @ s_a) / np.maximum(W @ s_b, 1e-9), (ka - kb)[msk].mean() / kb[msk].mean()

def asym(ka, kb):
    if NONE.sum() < CFG["MIN_N"] or FULL.sum() < CFG["MIN_N"]:
        return dict(gain_none_pct=np.nan, gain_full_pct=np.nan, asymmetry_pct=np.nan, ci_lo_pct=np.nan, ci_hi_pct=np.nan, p=np.nan)
    rn, pn = rel_reps(ka, kb, NONE); rf, pf = rel_reps(ka, kb, FULL)
    r = rn - rf; lo, hi = np.percentile(r, [2.5, 97.5])
    p = max(min(1.0, 2 * min((r <= 0).mean(), (r >= 0).mean())), 1.0 / len(r))
    return dict(gain_none_pct=100 * pn, gain_full_pct=100 * pf, asymmetry_pct=100 * (pn - pf), ci_lo_pct=100 * lo, ci_hi_pct=100 * hi, p=p)

rows = []
for (m_, v_, s_) in TS:
    for probe, a, b in [("edge probe: zero vs mirror edges", "edge_zero", "edge_mirror"), ("shift TTA: zero-filled vs centred", "tta_zero", "center"),
                        ("shift TTA: mirror-filled vs centred", "tta_mirror", "center")]:
        if (m_, v_, s_, a) in KLC and (m_, v_, s_, b) in KLC:
            r = asym(KLC[(m_, v_, s_, a)], KLC[(m_, v_, s_, b)])
            sh = (RAW[(m_, v_, s_, a)][FULL] - RAW[(m_, v_, s_, b)][FULL]).mean(0)
            rows.append(dict(model=m_, variant=v_, seed=s_, test=probe, **r, shift_Seizure=sh[0], shift_Other=sh[5], max_abs_shift=np.abs(sh).max()))
t2 = pd.DataFrame(rows)
if len(t2):
    t2["p_holm"] = np.nan; _m = t2["seed"].astype(str) == "mean"
    t2.loc[_m, "p_holm"] = holm(t2.loc[_m, "p"].to_numpy()) if _m.any() else np.nan   # Holm over the seed-averaged tests
t2.to_csv(f"{CFG['RESULTS_DIR']}/v9_boundary_tests.csv", index=False)
print(t2.round(4).to_string(index=False))

model        variant seed                                test  gain_none_pct  gain_full_pct  asymmetry_pct  ci_lo_pct  ci_hi_pct      p  shift_Seizure  shift_Other  max_abs_shift  p_holm
mamba   shift20_zero   42    edge probe: zero vs mirror edges       -16.9571        31.0909       -48.0480   -60.4096   -38.0537 0.0002        -0.2052       0.2236         0.2236     NaN
mamba   shift20_zero   42   shift TTA: zero-filled vs centred       -13.2261        -2.5957       -10.6304   -13.0197    -8.3730 0.0002        -0.0008       0.0017         0.0026     NaN
mamba   shift20_zero   42 shift TTA: mirror-filled vs centred        -0.2597        -2.5957         2.3359     0.6705     4.0855 0.0035        -0.0008       0.0017         0.0026     NaN
mamba   shift20_zero 1042    edge probe: zero vs mirror edges       -14.4148        30.5296       -44.9445   -55.6241   -35.8472 0.0002        -0.1799       0.2053         0.2053     NaN
mamba   shift20_zero 1042   shift TTA: zero-filled vs centred    

### Does shift augmentation change centred-view accuracy unevenly across annotation rounds?
`shift20_zero` and `shift20_mirror` against `shift0`, on the centred view, by context bin (negative = better than `shift0`).

In [13]:
rows = []
for m_, s_ in sorted({(k[0], k[2]) for k in TS}, key=str):
    base = KLC.get((m_, "shift0", s_, "center"))
    if base is None:
        continue
    for v_ in ("shift20_zero", "shift20_mirror"):
        k = KLC.get((m_, v_, s_, "center"))
        if k is None:
            continue
        for sname, msk in [("all EEGs", ALL), (">=10 votes", NV >= 10)] + [(b, CB == b) for b in CFG["CTX_LABELS"]]:
            mean, lo, hi, p = boot(k - base, msk)
            rows.append(dict(model=m_, seed=s_, comparison=f"{v_} - shift0", subset=sname, n=int(msk.sum()), diff=mean, ci_lo=lo, ci_hi=hi,
                             rel_pct=100 * mean / base[msk].mean(), p=p))
t3 = pd.DataFrame(rows)
if len(t3):
    t3["p_holm"] = np.nan; _m = t3["seed"].astype(str) == "mean"
    t3.loc[_m, "p_holm"] = holm(t3.loc[_m, "p"].to_numpy()) if _m.any() else np.nan   # Holm over the seed-averaged tests
t3.to_csv(f"{CFG['RESULTS_DIR']}/v9_shift_vs_center.csv", index=False)
print(t3.round(4).to_string(index=False))
print("outputs:", len(os.listdir(CFG["RESULTS_DIR"])), "files")

model seed              comparison             subset     n    diff   ci_lo   ci_hi  rel_pct      p  p_holm
  gru mean   shift20_zero - shift0           all EEGs 17089  0.0451  0.0355  0.0550   6.3742 0.0002   0.005
  gru mean   shift20_zero - shift0         >=10 votes  5600  0.0775  0.0685  0.0869  13.9974 0.0002   0.005
  gru mean   shift20_zero - shift0       none (<10 s)  9472  0.0853  0.0721  0.0985  13.4021 0.0002   0.005
  gru mean   shift20_zero - shift0 partial (10-140 s)  7126 -0.0034 -0.0129  0.0058  -0.4240 0.5000   1.000
  gru mean   shift20_zero - shift0     full (>=140 s)   491 -0.0275 -0.0474 -0.0088  -3.2620 0.0035   0.049
  gru mean shift20_mirror - shift0           all EEGs 17089  0.0049 -0.0006  0.0108   0.6961 0.0825   0.825
  gru mean shift20_mirror - shift0         >=10 votes  5600 -0.0028 -0.0088  0.0035  -0.4998 0.3735   1.000
  gru mean shift20_mirror - shift0       none (<10 s)  9472  0.0051 -0.0023  0.0128   0.8074 0.1825   1.000
  gru mean shift20_mirror - 